# Limpieza de datos.
## Grupo 2

### Importación de librerías.

In [1]:
import pandas as pd
import numpy as np

### Importación de datos.

In [2]:
df = pd.read_csv('train.csv', index_col=0)
df_labels = pd.read_csv('train_labels.csv', index_col=0)
labels = pd.read_csv('target_pairs.csv')

### Separando únicamente las variables usadas del conjunto train.

In [3]:
variables_usadas = set()
for x in labels['pair']:
    variables = x.split(' - ')
    for y in variables:
        variables_usadas.add(y)

variables_usadas = list(variables_usadas)
df = df[variables_usadas]

### Aplicando escala log a df.

In [4]:
df_log = np.log(df)
df_log.head(5)

,US_Stock_NEM_adj_close,FX_CHFJPY,US_Stock_XOM_adj_close,US_Stock_ALB_adj_close,FX_EURJPY,JPX_Platinum_Standard_Futures_Close,FX_EURAUD,US_Stock_URA_adj_close,FX_CADCHF,US_Stock_OKE_adj_close,...,FX_NOKJPY,FX_EURCHF,FX_NZDUSD,FX_AUDCAD,US_Stock_SCCO_adj_close,US_Stock_HES_adj_close,FX_AUDUSD,US_Stock_WMB_adj_close,FX_NOKUSD,US_Stock_BKR_adj_close
date_id,,,,,,,,,,,,,,,,,,,,,
0,3.416395,4.749668,4.095014,4.783912,4.908602,NaN,0.432021,2.559767,-0.252477,3.525210,...,2.626315,0.158934,-0.342165,-0.020610,3.553315,3.755879,-0.244121,3.023848,-2.094385,3.257966
1,3.407974,4.746975,4.114464,4.785887,4.907290,NaN,0.428469,2.561675,-0.250153,3.541362,...,2.631036,0.160315,-0.344013,-0.018001,3.555154,3.787758,-0.244905,3.051091,-2.092689,3.297510
2,3.419011,4.751435,4.115848,4.773903,4.913950,8.139441,0.428585,2.569286,-0.248223,3.553177,...,2.637891,0.162515,-0.334396,-0.017847,3.550040,3.811562,-0.240198,3.068132,-2.087675,3.329791
3,3.422665,4.753389,4.115041,4.785356,4.912255,8.156510,0.423941,2.569286,-0.241653,3.550683,...,2.641595,0.158866,-0.332682,-0.023423,3.557805,3.842122,-0.238941,3.065366,-2.085661,3.324288
4,3.421621,4.751550,4.119526,4.827938,4.908758,NaN,0.422403,2.557212,-0.239192,3.560975,...,2.640040,0.157208,-0.331676,-0.026003,3.562477,3.848429,-0.242316,3.069057,-2.088628,3.326319


### Seprando las etiquetas del conjunto lables.

In [5]:
labels['etiquetas'] = [x.split(' - ') for x in labels['pair']]

### Función que permite reemplazar valores nulos de df_labels mediante cálculo de df_log.

In [6]:
def nulos_calculables():

    # Identifica cuántos valores fueron alterados.
    contador = 0

    # Dimensiones.
    n, m = df_labels.shape

    # Se recorren todas las columnas a operar
    for columna in df_labels.columns:

        # Explicación de cómo calcular los datos.
        variables = labels.loc[labels['target'] == columna, 'etiquetas'].iloc[0]
        lag = labels.loc[labels['target'] == columna, 'lag'].iloc[0]

        # Se evalúa si es requerido y posible un cambio.
        for i in range(1, n-lag+1):
            if pd.isna(df_labels.loc[i, columna]):

                filas_necesarias = [i-1, i+lag-1]
                if df_log.loc[filas_necesarias, variables].notna().all().all():

                    # Se realiza el cambio.
                    parte_1 = df_log.loc[i+lag-1, variables[0]]- df_log.loc[i-1, variables[0]]
                    if len(variables) == 2:
                        parte_2 = df_log.loc[i+lag-1, variables[1]]- df_log.loc[i-1, variables[1]]
                        df_labels.loc[i, columna] = parte_1 - parte_2
                    else:
                        df_labels.loc[i, columna] = parte_1

                    # Aumenta el contador.
                    contador += 1

    print('Se realizaron', contador, 'cambios a valores nulos.')
nulos_calculables()

Se realizaron 66977 cambios a valores nulos.


### Función que permite procesar los valores nulos al inicio y al final de df_log.

In [7]:
def nulos_inicio_fin():

    cambios = 0
    n, m = df_log.shape

    # Procesando si el primer valor es nulo
    for columna in df_log.columns:
        contador = 0

        if pd.isna(df_log.loc[contador, columna]):
            while pd.isna(df_log.loc[contador, columna]):
                contador += 1
            valor_util = df_log.loc[contador, columna]
            for j in range(0, contador):
                df_log.loc[j, columna] = valor_util
                cambios += 1

    # Procesando si el último valor es nulo
    for columna in df_log.columns:
        contador = 1

        if pd.isna(df_log.loc[n-contador, columna]):
            while pd.isna(df_log.loc[n-contador, columna]):
                contador += 1
            valor_util = df_log.loc[n-contador, columna]
            for j in range(1, contador):
                df_log.loc[n-j, columna] = valor_util
                cambios += 1

    print('Se realizaron', cambios, 'cambios a valores nulos.')
nulos_inicio_fin()

Se realizaron 36 cambios a valores nulos.


### Función que permite procesar los valores nulos al inicio y al final de df_labels.

In [8]:
def nulos_inicio_fin_2():

    cambios = 0
    n, m = df_labels.shape

    # Procesando si el primer valor es nulo
    for columna in df_labels.columns:
        contador = 0

        if pd.isna(df_labels.loc[contador, columna]):
            while pd.isna(df_labels.loc[contador, columna]):
                contador += 1
            valor_util = df_labels.loc[contador, columna]
            for j in range(0, contador):
                df_labels.loc[j, columna] = valor_util
                cambios += 1

    # Procesando si el último valor es nulo
    for columna in df_labels.columns:
        contador = 1

        if pd.isna(df_labels.loc[n-contador, columna]):
            while pd.isna(df_labels.loc[n-contador, columna]):
                contador += 1
            valor_util = df_labels.loc[n-contador, columna]
            for j in range(1, contador):
                df_labels.loc[n-j, columna] = valor_util
                cambios += 1

    print('Se realizaron', cambios, 'cambios a valores nulos.')
nulos_inicio_fin_2()

Se realizaron 319 cambios a valores nulos.


### Procesar mediante interpolación, los valores faltantes del conjunto df_log.

In [9]:
nans_iniciales = df_log.isna().sum().sum()
df_log = df_log.interpolate(method="linear", axis=0)
nans_finales = df_log.isna().sum().sum()

print('Se realizaron', nans_iniciales - nans_finales, 'cambios a valores nulos.')
print('Valores nulos actuales:', nans_finales)

Se realizaron 4919 cambios a valores nulos.
Valores nulos actuales: 0


In [10]:
df_log.head(3)

,US_Stock_NEM_adj_close,FX_CHFJPY,US_Stock_XOM_adj_close,US_Stock_ALB_adj_close,FX_EURJPY,JPX_Platinum_Standard_Futures_Close,FX_EURAUD,US_Stock_URA_adj_close,FX_CADCHF,US_Stock_OKE_adj_close,...,FX_NOKJPY,FX_EURCHF,FX_NZDUSD,FX_AUDCAD,US_Stock_SCCO_adj_close,US_Stock_HES_adj_close,FX_AUDUSD,US_Stock_WMB_adj_close,FX_NOKUSD,US_Stock_BKR_adj_close
date_id,,,,,,,,,,,,,,,,,,,,,
0,3.416395,4.749668,4.095014,4.783912,4.908602,8.139441,0.432021,2.559767,-0.252477,3.525210,...,2.626315,0.158934,-0.342165,-0.020610,3.553315,3.755879,-0.244121,3.023848,-2.094385,3.257966
1,3.407974,4.746975,4.114464,4.785887,4.907290,8.139441,0.428469,2.561675,-0.250153,3.541362,...,2.631036,0.160315,-0.344013,-0.018001,3.555154,3.787758,-0.244905,3.051091,-2.092689,3.297510
2,3.419011,4.751435,4.115848,4.773903,4.913950,8.139441,0.428585,2.569286,-0.248223,3.553177,...,2.637891,0.162515,-0.334396,-0.017847,3.550040,3.811562,-0.240198,3.068132,-2.087675,3.329791


### Se calculan los faltantes de df_labels a partir de df_log.

In [11]:
nulos_calculables()

Se realizaron 20107 cambios a valores nulos.


In [15]:
print('Valores nulos actuales:', df_labels.isna().sum().sum())

Valores nulos actuales: 0


### Guardando los conjuntos limpios.

In [16]:
df_log.to_csv('entrenamiento.csv')
df_labels.to_csv('entrenamiento_etiquetas.csv')